# 00 — Ingest

Pull raw StatsBomb open data for **Premier League 2015/16**
(`competition_id=2`, `season_id=27`), 380 matches. Writes:

- `data/statsbomb.csv` — raw `statsbombpy` events, one row per event, with a `match_id` column
- `data/event_df.csv` — SPADL actions (`socceraction`), with action-type / result / team / player names

Hits the network; takes a while.

In [1]:
import warnings
from statsbombpy.api_client import NoAuthWarning

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

import pandas as pd
from statsbombpy import sb
from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl

COMPETITION_ID = 2
SEASON_ID = 27

api = StatsBombLoader(getter="remote", creds=None)

fixtures = api.games(competition_id=COMPETITION_ID, season_id=SEASON_ID)
match_ids = list(fixtures["game_id"].unique())
print(f"{len(match_ids)} matches")

380 matches


## Raw `statsbombpy` events

In [2]:
raw = []
for i, match_id in enumerate(match_ids, 1):
    events = sb.events(match_id=match_id)
    events["match_id"] = match_id
    raw.append(events)
    print(f"  raw events {i}/{len(match_ids)}  match {match_id}  ({len(events)} rows)")

df = pd.concat(raw, ignore_index=True)
df.to_csv("data/statsbomb.csv")
print(f"data/statsbomb.csv  {df.shape}")

  raw events 1/380  match 3754217  (3732 rows)


  raw events 2/380  match 3754117  (3451 rows)


  raw events 3/380  match 3754296  (3858 rows)


  raw events 4/380  match 3753983  (3715 rows)


  raw events 5/380  match 3754160  (3489 rows)


  raw events 6/380  match 3754129  (4090 rows)


... (371 outputs left out)

  raw events 378/380  match 3754141  (3564 rows)


  raw events 379/380  match 3754128  (3711 rows)


  raw events 380/380  match 3754078  (3352 rows)


data/statsbomb.csv  (1313773, 118)


## SPADL actions (`socceraction`)

In [3]:
events = []
for i, match_id in enumerate(match_ids, 1):
    event = api.events(match_id)
    home_id = fixtures[fixtures["game_id"] == match_id]["home_team_id"].item()

    df_actions = spadl.statsbomb.convert_to_actions(event, home_team_id=home_id)
    df_actions = (
        spadl.add_names(df_actions)  # add actiontype and result names
        .merge(api.teams(game_id=match_id))  # add team names
        .merge(api.players(game_id=match_id))  # add player names
    )

    events.append(df_actions)
    print(f"  spadl actions {i}/{len(match_ids)}  match {match_id}  ({len(df_actions)} rows)")

event_df = pd.concat(events, ignore_index=True)
event_df.to_csv("data/event_df.csv")
print(f"data/event_df.csv  {event_df.shape}")

/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool

/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/spadl/statsbomb.py:53: UserWarning: Inferred xy_fidelity_version=2. If this is incorrect, please specify the correct version using the xy_fidelity_version argument
  warnings.warn(
/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/spadl/statsbomb.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  events["extra"].fillna({}, inplace

/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool

  spadl actions 1/380  match 3754217  (2161 rows)


/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool

  spadl actions 2/380  match 3754117  (1978 rows)


... (778 outputs left out)

  spadl actions 379/380  match 3754128  (2124 rows)


/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:337: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["under_pressure"] = eventsdf["under_pressure"].fillna(False).astype(bool)
/Users/albertezem/Developer/oracle/.claude/worktrees/rerun-notebooks-f72421/.venv/lib/python3.12/site-packages/socceraction/data/statsbomb/loader.py:338: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eventsdf["counterpress"] = eventsdf["counterpress"].fillna(False).astype(bool

  spadl actions 380/380  match 3754078  (1942 rows)


data/event_df.csv  (758434, 25)
